# Unit / drum grouping

Builds two JSON files from the HPCL tag sheets, for drum-level grouping (e.g. drum-wise PCA later):

- `drum_tags.json`: `{drum: {"process_tags": [...]}}`
- `unit_drums.json`: `{unit: [drum, ...]}` (new unit names)

Sections: **0 Config, 1 Inspect, 2 Rename units, 3 Build JSONs, 4 Validate, 5 Save**.
Raw parquet files are never modified; unit renames are applied in memory (`PARQUET_INDEX`, `load_unit`).

Control valves (`All_unit_control_valve.xlsx`) are left out for now: none of their tags has a parquet file.
Only tags present in the parquet data are kept; drums and units left with no tags are dropped.

## 0. Config

In [1]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)

OUT_DIR = Path.cwd()                       # JSONs are saved next to this notebook
DATA_DIR = (OUT_DIR / ".." / "Dataset_HPCL").resolve()
TAGS_XLSX = DATA_DIR / "FINAL_SHEETS.xlsx"
NON_UNIT_DIRS = {"flare_data"}             # flare targets, not a process unit

# Step 1 rename map (old -> new). Keys are matched ignoring "_", "-", " " and case,
# so "ISOM_4" (folder/sheet) hits "ISOM4".
UNIT_RENAME = {
    "ISOM2": "LPG",
    "ISOM4": "DHT",
    "NSU_NHT_CCR": "CCR",
    "APS_NVPS_LEU": "CDU",
}

# Build options
KEEP_ONLY_PARQUET_TAGS = True   # process_tags hold only tags present in the parquet data
DROP_UNITS_WITHOUT_PARQUET = True  # Excel sheets with no parquet folder are left out of the JSONs

# Cell values in the instrument columns that mean "no tag"
PLACEHOLDERS = {"", "NAN", "NIL", "NOT AVAILABLE", "**", ".PV", "N/A", "N/A.PV", "NA", "-"}

## 1. Inspect

In [2]:
def compact(name):
    """Comparison key for unit labels: upper case, no separators."""
    return re.sub(r"[\s_\-/]+", "", str(name)).upper()


def tag_key(tag):
    """Comparison key for tags: upper case, no whitespace, PHD read suffix removed."""
    t = re.sub(r"\s+", "", str(tag)).upper()
    for suffix in (".DACA.PV", ".PIDA.PV", ".PV", ".DACA"):
        if t.endswith(suffix):
            return t[: -len(suffix)]
    return t


# --- parquet folder: one file per tag, columns = [Timestamp, <tag>] ---
pq_rows = []
for f in sorted(DATA_DIR.glob("*/*.parquet")):
    cols = [c for c in pq.read_schema(f).names if c != "Timestamp"]
    for c in cols:
        pq_rows.append({"folder": f.parent.name, "file": f.name, "column": c})
pq_all = pd.DataFrame(pq_rows)
parquet_tags = pq_all[~pq_all.folder.isin(NON_UNIT_DIRS)].copy()
parquet_tags["key"] = parquet_tags.column.map(tag_key)

print("Parquet folders (files):")
print(pq_all.folder.value_counts().to_string())
print("\nNon-unit folders:", {d: pq_all[pq_all.folder == d].column.tolist() for d in NON_UNIT_DIRS})
assert parquet_tags.key.is_unique, "two parquet columns collapse to the same tag key"
KEY_TO_PARQUET = dict(zip(parquet_tags.key, parquet_tags.column))
KEY_TO_FOLDER = dict(zip(parquet_tags.key, parquet_tags.folder))

Parquet folders (files):
folder
LOUP            79
NSU_NHT_CCR     78
ISOM            45
PG              38
PDA             28
SEU1            23
DIU             18
SEU2            17
APS_NVPS_LEU    14
ISOM_3          14
SEU3            14
DAK              5
flare_data       4
ISOM_4           3

Non-unit folders: {'flare_data': ['(44ft2505.pv*509mw5101.daca.pv/22.414)/1000', '(LRflare.pv+LREflare.pv+99fi4601.daca.pv)/1000', '509fi5201.daca.pv/1000', '509mw5101.daca.pv']}


In [3]:
# --- tags workbook: one sheet per unit; header row is the one holding "Description" ---
def read_tag_sheet(path, sheet):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    hdr = next(i for i in range(6) if "Description" in [str(v).strip() for v in raw.iloc[i]])
    df = raw.iloc[hdr + 1 :].copy()
    df.columns = [str(c).strip() for c in raw.iloc[hdr]]
    banner = str(raw.iloc[0, 3]).strip() if hdr > 0 else ""
    return df.dropna(how="all"), hdr, banner


tags_book = pd.ExcelFile(TAGS_XLSX)
sheet_info, sheets = [], {}
for s in tags_book.sheet_names:
    df, hdr, banner = read_tag_sheet(TAGS_XLSX, s)
    sheets[s] = df
    sheet_info.append({"sheet": s, "header_row": hdr, "banner (unit label in sheet)": banner,
                       "rows": len(df), "columns": list(df.columns)[:7]})
print("Tags workbook:", TAGS_XLSX.name)
display(pd.DataFrame(sheet_info))
display(sheets["SEU3"].head(3))

Tags workbook: FINAL_SHEETS.xlsx


,sheet,header_row,banner (unit label in sheet),rows,columns
0,SEU3,1,SEU1,37,"[S.No, PSV Tag/BR No., PSV Set Pressure, kg/cm2g, Descri..."
1,SEU1,1,SEU1,32,"[S.No, PSV Tag/BR No., PSV Set Pressure, kg/cm2g, Descri..."
2,SEU2,1,SEU2,23,"[S.No, PSV Tag/Br No, PSV Set Pressure, kg/cm2g, Descrip..."
3,DIU,1,NHGU,15,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."
4,ISOM,1,ISOM,58,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."
5,PDA,0,,63,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."
6,LOUP,0,,66,"[S.No, PSV Tag, Set Point, Description, Molecular Weight..."
7,ISOM_2,1,LPG Mounded Bullets,12,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."
8,PG,1,Prime G,35,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."
9,ISOM_3,1,"OFCCU, GCU, Propane and Vr LPG treating",17,"[S.No, PSV Tag, PSV Set Pressure, kg/cm2g, Description, ..."


,S.No,PSV Tag/BR No.,"PSV Set Pressure, kg/cm2g",Description,Molecular Weight of Stream,Pressure Instrument,Temperature Instrument
2,1,1412,22,P4453A/B,NaN,NaN,NaN
3,2,SV-4205,16.5,P4201,NaN,NaN,NaN
4,4,4202,3.86,D4201,HC + NMP + H2O,42PC4215.PV,NaN


## 2. Rename units

In [4]:
RENAME_BY_KEY = {compact(k): v for k, v in UNIT_RENAME.items()}
FOLDERS = sorted(set(parquet_tags.folder))
FOLDER_BY_KEY = {compact(f): f for f in FOLDERS}


def canonical_old(label):
    """Map any old-style label (sheet name, folder) to the parquet folder spelling."""
    k = compact(label)
    if k in FOLDER_BY_KEY:
        return FOLDER_BY_KEY[k]
    return str(label).strip().replace(" ", "_").replace("-", "_")


def new_unit(label):
    """Old or new unit label -> new unit name."""
    old = canonical_old(label)
    return RENAME_BY_KEY.get(compact(old), old)


# Collision check: a new name must not equal any other existing unit label
existing = set(FOLDERS) | {canonical_old(s) for s in tags_book.sheet_names}
existing_keys = {compact(e): e for e in existing}
for old, new in UNIT_RENAME.items():
    clash = existing_keys.get(compact(new))
    status = f"COLLIDES with existing '{clash}'" if clash and compact(clash) != compact(old) else "ok"
    print(f"{old:>13} -> {new:<4} {status}")

unit_table = pd.DataFrame(
    [{"label": l, "source": src, "canonical_old": canonical_old(l), "new": new_unit(l)}
     for src, labels in [("parquet folder", FOLDERS), ("tags sheet", tags_book.sheet_names)]
     for l in labels]
)
display(unit_table)
assert not (set(unit_table.new) & {"ISOM2", "ISOM4", "NSU_NHT_CCR", "APS_NVPS_LEU"})

# Parquet index with new unit names (raw files untouched)
PARQUET_INDEX = parquet_tags.assign(unit=parquet_tags.folder.map(new_unit))[["unit", "folder", "file", "column"]]


def load_unit(unit, columns=None):
    """Load one unit (new name) as a wide Timestamp-indexed frame. Renames happen here, in memory."""
    rows = PARQUET_INDEX[PARQUET_INDEX.unit == unit]
    if columns is not None:
        rows = rows[rows.column.isin(columns)]
    frames = [pd.read_parquet(DATA_DIR / r.folder / r.file).set_index("Timestamp") for r in rows.itertuples()]
    return pd.concat(frames, axis=1) if frames else pd.DataFrame()

        ISOM2 -> LPG  ok
        ISOM4 -> DHT  ok
  NSU_NHT_CCR -> CCR  ok
 APS_NVPS_LEU -> CDU  ok


,label,source,canonical_old,new
0,APS_NVPS_LEU,parquet folder,APS_NVPS_LEU,CDU
1,DAK,parquet folder,DAK,DAK
2,DIU,parquet folder,DIU,DIU
3,ISOM,parquet folder,ISOM,ISOM
4,ISOM_3,parquet folder,ISOM_3,ISOM_3
5,ISOM_4,parquet folder,ISOM_4,DHT
6,LOUP,parquet folder,LOUP,LOUP
7,NSU_NHT_CCR,parquet folder,NSU_NHT_CCR,CCR
8,PDA,parquet folder,PDA,PDA
9,PG,parquet folder,PG,PG


## 3. Build JSONs

In [5]:
def clean_drum(name):
    return re.sub(r"\s+", " ", str(name)).strip()


def split_cell(cell):
    """Instrument cell -> list of raw tags. Expands '14PT416XA/XB.PV' and '102PI4457/102PI4557'."""
    s = re.sub(r"\s+", "", str(cell))
    if s.upper() in PLACEHOLDERS:
        return []
    if "/" not in s:
        return [s]
    first, *rest = s.split("/")
    m = re.match(r"^(.*?)(\.[A-Za-z.]+)?$", first)
    stem, suffix = m.group(1), m.group(2) or ""
    out = [first]
    for r in rest:
        if re.match(r"^\d+[A-Za-z]", r):          # full tag
            out.append(r)
        else:                                       # tail only, e.g. 'XB.PV'
            r_stem = r.split(".")[0]
            out.append(stem[: len(stem) - len(r_stem)] + r + ("" if "." in r else suffix))
    return out


rows = []
for sheet, df in sheets.items():
    unit = new_unit(sheet)
    for _, r in df.iterrows():
        drum = clean_drum(r.get("Description", ""))
        for col in ("Pressure Instrument", "Temperature Instrument"):
            if col not in df.columns or pd.isna(r[col]):
                continue
            for raw in split_cell(r[col]):
                k = tag_key(raw)
                rows.append({"unit": unit, "sheet": sheet, "drum": drum, "excel_tag": raw, "source_col": col,
                             "parquet_tag": KEY_TO_PARQUET.get(k), "parquet_folder": KEY_TO_FOLDER.get(k)})
tag_rows = pd.DataFrame(rows)
tag_rows = tag_rows[tag_rows.drum.ne("") & tag_rows.drum.str.lower().ne("nan")]
# drum names differing only in case within a unit are the same drum: keep the first spelling
case_variants = (tag_rows.assign(k=tag_rows.drum.str.upper()).groupby(["unit", "k"]).drum.unique())
case_variants = [list(v) for v in case_variants if len(v) > 1]
tag_rows["drum"] = tag_rows.groupby(["unit", tag_rows.drum.str.upper()]).drum.transform("first")
print("Drum names merged (case-only differences):", case_variants)
tag_rows["parquet_unit"] = tag_rows.parquet_folder.map(lambda f: new_unit(f) if isinstance(f, str) else None)

# Summary required by Step 0
summ = (tag_rows.groupby("unit")
        .agg(sheet=("sheet", "first"), drums=("drum", "nunique"), tag_mentions=("excel_tag", "size"),
             unique_tags=("excel_tag", lambda s: s.map(tag_key).nunique()),
             unique_tags_in_parquet=("parquet_tag", "nunique")))
summ["duplicate_mentions"] = summ.tag_mentions - summ.unique_tags
summ["parquet_cols_in_folder"] = summ.index.map(PARQUET_INDEX.unit.value_counts()).fillna(0).astype(int)
display(summ)

tags_per_drum = tag_rows.groupby(["unit", "drum"]).excel_tag.apply(lambda s: s.map(tag_key).nunique())
print("Tags per drum (unique, Excel side): min/median/max =",
      tags_per_drum.min(), tags_per_drum.median(), tags_per_drum.max())

cross = tag_rows[tag_rows.parquet_unit.notna() & tag_rows.parquet_unit.ne(tag_rows.unit)]
print("\nExcel tags found in a different unit's parquet folder:", len(cross))
if len(cross):
    display(cross)

excel_missing = (tag_rows[tag_rows.parquet_tag.isna()]
                 .groupby("unit").excel_tag.apply(lambda s: sorted(set(s))))
print("\nExcel tags with no parquet column:")
for u, t in excel_missing.items():
    print(f"  {u:<12} ({len(t)}): {t}")
used_keys = set(tag_rows.excel_tag.map(tag_key))
pq_unused = PARQUET_INDEX[~PARQUET_INDEX.column.map(tag_key).isin(used_keys)]
print("\nParquet columns not referenced in the tags Excel:", len(pq_unused))
if len(pq_unused):
    display(pq_unused)

Drum names merged (case-only differences): [['Fuel gas mixing drum to Header', 'Fuel Gas mixing drum to Header']]


,sheet,drums,tag_mentions,unique_tags,unique_tags_in_parquet,duplicate_mentions,parquet_cols_in_folder
unit,,,,,,,
CCR,NSU NHT-CCR,67,182,95,78,87,78
CDU,APS_NVPS_LEU,21,47,40,14,7,14
CPP,CPP,7,19,4,0,15,0
DAK,DAK,7,12,6,5,6,5
DHT,ISOM_4,28,79,41,3,38,3
DIU,DIU,13,21,21,18,0,18
ISOM,ISOM,35,116,71,45,45,45
ISOM_3,ISOM_3,17,30,27,14,3,14
LOUP,LOUP,56,103,96,79,7,79


Tags per drum (unique, Excel side): min/median/max = 1 2.0 6

Excel tags found in a different unit's parquet folder: 0

Excel tags with no parquet column:
  CCR          (17): ['101TI1031.DACA.PV', '102PC0501B.DACA.PV', '102PC4602R.DAA.PV', '102PI6710N.PV', '102PI6720N.PV', '102PI6730N.PV', '102PI6740N.PV', '102PI6750N.PV', '102PI6760N.PV', '102PI8501N.DACA.PV', '102TI1153.DACA.PV', '102TI1154.DACA.PV', '102TI4702N.DACA.PV', '102TI7804.DACA.PV', 'Notavailable', 'TI7407.DACA.PV', 'TI7517.DACA.PV']
  CDU          (26): ['11PC1803.PIDA.PV', '11PC1804.PIDA.PV', '11PC3101.PIDA.PV', '11PC4711.PIDA.PV', '11PC701A.PIDA.PV', '11PC7801.PIDA.PV', '11TC1801.PIDA.PV', '11TC1805.DACA.PV', '11TC401.PIDA.PV', '24PC3501.PIDA.PV', '24PI1613.AI.PV.VALUE', '24PI3012.AI.PV.VALUE', '24PI3013.AI.PV.VALUE', '24PI3014.AI.PV.VALUE', '24PI3103.AI.PV.VALUE', '24PI3104.AI.PV.VALUE', '24PI3105.AI.PV.VALUE', '24PI3205.AI.PV.VALUE', '24PI3207.AI.PV.VALUE', '24PI3302.AI.PV.VALUE', '24PI3425.AI.PV.VALUE', '24TI2309.AI.

In [6]:
# --- assemble ---
in_parquet_units = set(PARQUET_INDEX.unit)
proc_ok = tag_rows[tag_rows.parquet_tag.notna()] if KEEP_ONLY_PARQUET_TAGS else tag_rows
if DROP_UNITS_WITHOUT_PARQUET:
    dropped_units = sorted(set(tag_rows.unit) - in_parquet_units)
    proc_ok = proc_ok[proc_ok.unit.isin(in_parquet_units)]
else:
    dropped_units = []

drum_order = tag_rows[["unit", "drum"]].drop_duplicates()          # Excel order
# drum name used in more than one unit -> "<unit>__<drum>" everywhere
drum_units = drum_order.groupby("drum").unit.nunique()
AMBIGUOUS_DRUMS = sorted(drum_units[drum_units > 1].index)
drum_key = lambda u, d: f"{u}__{d}" if d in AMBIGUOUS_DRUMS else d

drum_tags, unit_drums, empty_drums = {}, defaultdict(list), []
for u, d in drum_order.itertuples(index=False):
    if DROP_UNITS_WITHOUT_PARQUET and u not in in_parquet_units:
        continue
    sel = proc_ok[(proc_ok.unit == u) & (proc_ok.drum == d)]
    ptags = list(dict.fromkeys(sel.parquet_tag if KEEP_ONLY_PARQUET_TAGS else
                               sel.parquet_tag.fillna(sel.excel_tag)))
    if not ptags:
        empty_drums.append((u, d))
        continue
    key = drum_key(u, d)
    drum_tags[key] = {"process_tags": ptags}
    unit_drums[u].append(key)
unit_drums = dict(unit_drums)

print("Drum names shared by several units (keyed '<unit>__<drum>'):", AMBIGUOUS_DRUMS)
print("Units left out (no parquet data):", dropped_units)
print(f"Drums left out (no tag present in parquet): {len(empty_drums)}")
for u, d in empty_drums:
    print(f"   {u:<12} {d}")
print(f"\nResult: {len(unit_drums)} units, {len(drum_tags)} drums")

Drum names shared by several units (keyed '<unit>__<drum>'): []
Units left out (no parquet data): ['CPP', 'LPG', 'PDU']
Drums left out (no tag present in parquet): 74
   SEU3         D4210
   SEU3         D4215
   SEU3         E5207 O/L
   SEU3         E5207 to P5205
   SEU3         E5207
   SEU3         E4614
   SEU3         R5201
   SEU1         C401D D/C
   SEU1         C502 LO Pump for turbine pump
   SEU1         C502 LO Pump for Motor pump
   SEU2         D3204
   SEU2         Recovery Bypass Line
   DIU          D18 - Wash water drum
   ISOM         DIP tower
   ISOM         MUG KOD
   ISOM         MUG-A inter stage KOD
   ISOM         MUG-B inter stage KOD
   ISOM         HPP Rx Outlet
   ISOM         ISOMERISATION REACTOR (103-R-6001 ) Inlet
   PDA          C4101A
   PDA          C4101B
   PDA          C4102B
   PDA          C3 to C4101A
   PDA          C3 to C4101B
   PDA          P4101A(W) discharge
   PDA          P4101B E discharge
   PDA          V4122 overhead
   PDA    

## 4. Validate

In [7]:
all_pq_cols = set(PARQUET_INDEX.column)
json_tags = [(k, t) for k, v in drum_tags.items() for t in v["process_tags"]]

missing = sorted({t for _, t in json_tags if t not in all_pq_cols})
print(f"1) JSON tags with no parquet column: {len(missing)}", missing)

unassigned_pq = PARQUET_INDEX[~PARQUET_INDEX.column.isin({t for _, t in json_tags})]
print(f"\n2) Parquet tags not assigned to any drum: {len(unassigned_pq)}")
display(unassigned_pq)

owners = defaultdict(list)
for k, t in json_tags:
    owners[t].append(k)
multi = {t: ks for t, ks in owners.items() if len(set(ks)) > 1}
print(f"\n3) Tags assigned to more than one drum: {len(multi)}")
display(pd.DataFrame([{"tag": t, "n_drums": len(ks), "drums": " | ".join(ks)} for t, ks in sorted(multi.items())]))

seen = Counter(k for ks in unit_drums.values() for k in ks)
assert all(v == 1 for v in seen.values()), "a drum key is listed under more than one unit"
assert set(seen) == set(drum_tags), "unit_drums and drum_tags keys differ"
print("\nEach drum key appears under exactly one unit; both JSONs share the same drum keys.")

1) JSON tags with no parquet column: 0 []

2) Parquet tags not assigned to any drum: 0


,unit,folder,file,column



3) Tags assigned to more than one drum: 54


,tag,n_drums,drums
0,02PC201B.PV,2,P201A | P201
1,02PDI215.PV,2,R201 | GR201
2,02TC205W.PV,2,P201A | P201
3,05PC501.PV,2,LPG Make up to C502 | D502
4,05PC502.PV,2,D/S of C502 Recycle | D503
5,05TI509.PV,2,LPG Make up to C502 | D503
6,102PC4702.DACA.PV,5,Recontact Drum 102D2004 Net Gas | Recontact Drum LFLF | ...
7,102PC5101R.DACA.PV,3,Stabilizer Reciever Drum | Stabilizer Condenser 102E2009...
8,102PC6702.DACA.PV,2,102FIL4053N Hydrogen Filter | 102FIL4054N Hydrogen Filter
9,102PI4244AN.PV,2,1st Stage Suction cooler 102E2003A inlet | 1st Stage Suc...



Each drum key appears under exactly one unit; both JSONs share the same drum keys.


In [8]:
per_drum = pd.DataFrame([
    {"unit": u, "drum": k, "tags": len(drum_tags[k]["process_tags"])}
    for u, ks in unit_drums.items() for k in ks])
per_unit = per_drum.groupby("unit", sort=False).agg(
    drums=("drum", "size"), tag_slots=("tags", "sum"),
    tags_per_drum_median=("tags", "median"), single_tag_drums=("tags", lambda x: int((x == 1).sum())))
per_unit["unique_tags"] = [len({t for k in unit_drums[u] for t in drum_tags[k]["process_tags"]}) for u in per_unit.index]
per_unit["parquet_cols"] = per_unit.index.map(PARQUET_INDEX.unit.value_counts()).fillna(0).astype(int)
total = per_unit.sum(numeric_only=True)
total["tags_per_drum_median"] = per_drum.tags.median()
per_unit.loc["TOTAL"] = total
per_unit = per_unit.astype({c: int for c in per_unit.columns if c != "tags_per_drum_median"})
print("Per-unit counts")
display(per_unit)
print("Per-drum counts")
display(per_drum)

Per-unit counts


,drums,tag_slots,tags_per_drum_median,single_tag_drums,unique_tags,parquet_cols
unit,,,,,,
SEU3,17,20,1.0,14,14,14
SEU1,24,29,1.0,19,23,23
SEU2,16,19,1.0,13,17,17
DIU,12,18,1.5,6,18,18
ISOM,29,45,2.0,13,45,45
PDA,36,67,2.0,5,28,28
LOUP,51,82,2.0,24,79,79
PG,16,38,2.0,1,38,38
ISOM_3,14,17,1.0,11,14,14


Per-drum counts


,unit,drum,tags
0,SEU3,D4201,1
1,SEU3,T4201,1
2,SEU3,D4205,1
3,SEU3,D4207,1
4,SEU3,D4204,1
...,...,...,...
296,CDU,Amine tower 24-T-1005 O/H (East),1
297,CDU,Ejector condensate drum 24-D-1004,1
298,CDU,17T1 O/H Line,2
299,CDU,17T2 O/H Line,2


## 5. Save

In [9]:
(OUT_DIR / "drum_tags.json").write_text(json.dumps(drum_tags, indent=2))
(OUT_DIR / "unit_drums.json").write_text(json.dumps(unit_drums, indent=2))
print("Saved:", OUT_DIR / "drum_tags.json", "|", OUT_DIR / "unit_drums.json")

print("\ndrum_tags.json (first 4):")
print(json.dumps(dict(list(drum_tags.items())[:4]), indent=2))
print("\nunit_drums.json (first 3):")
print(json.dumps({u: unit_drums[u] for u in list(unit_drums)[:3]}, indent=2))

Saved: /Users/gyan/Documents/Root-Cause-Detection/unit_drum_grouping/drum_tags.json | /Users/gyan/Documents/Root-Cause-Detection/unit_drum_grouping/unit_drums.json

drum_tags.json (first 4):
{
  "D4201": {
    "process_tags": [
      "42PC4215.PV"
    ]
  },
  "T4201": {
    "process_tags": [
      "42PC4204.PV"
    ]
  },
  "D4205": {
    "process_tags": [
      "42PI4213.PV"
    ]
  },
  "D4207": {
    "process_tags": [
      "42PC4220.PV"
    ]
  }
}

unit_drums.json (first 3):
{
  "SEU3": [
    "D4201",
    "T4201",
    "D4205",
    "D4207",
    "D4204",
    "D4208",
    "T4207",
    "K4201 D/C",
    "K5201 D/C",
    "C4502 discharge",
    "D4502",
    "D4503",
    "D5204",
    "T4202",
    "V4301",
    "D/S of PIC4501",
    "T4207 North or South"
  ],
  "SEU1": [
    "D403",
    "K604 I stage",
    "K604 II stage",
    "C401C D/C",
    "D209",
    "D201",
    "T201",
    "D205",
    "D207",
    "D204",
    "D208",
    "R201",
    "E207 O/L",
    "E207 to P205",
    "LPG Make up to

## 6. Final data stats

In [10]:
# Coverage of the kept tags in the raw data (reads each kept parquet file once)
kept = PARQUET_INDEX[PARQUET_INDEX.column.isin({t for v in drum_tags.values() for t in v["process_tags"]})]
stats = []
for r in kept.itertuples():
    s_ = pd.read_parquet(DATA_DIR / r.folder / r.file)
    stats.append({"unit": r.unit, "tag": r.column, "rows": len(s_), "start": s_.Timestamp.min(),
                  "end": s_.Timestamp.max(), "nan_frac": s_[r.column].isna().mean()})
stats = pd.DataFrame(stats)
cov = stats.groupby("unit", sort=False).agg(tags=("tag", "size"), rows_min=("rows", "min"), rows_max=("rows", "max"),
                                            start=("start", "min"), end=("end", "max"), nan_frac_mean=("nan_frac", "mean"))
cov = cov.join(per_unit[["drums"]]).loc[list(unit_drums)][["drums", "tags", "rows_min", "rows_max", "start", "end", "nan_frac_mean"]]
display(cov.round({"nan_frac_mean": 4}))
print(f"Units: {len(unit_drums)} | drums: {len(drum_tags)} | unique tags: {kept.column.nunique()} "
      f"| tag-drum slots: {sum(len(v['process_tags']) for v in drum_tags.values())}")
print(f"Overall time range: {stats.start.min()} -> {stats.end.max()} | rows per tag: {stats.rows.min()}-{stats.rows.max()}")
print("Units dropped (no parquet data):", dropped_units)

,drums,tags,rows_min,rows_max,start,end,nan_frac_mean
unit,,,,,,,
SEU3,17,14,581774,581774,2025-04-12,2026-05-21,0.0
SEU1,24,23,581774,581774,2025-04-12,2026-05-21,0.0
SEU2,16,17,581774,581774,2025-04-12,2026-05-21,0.0
DIU,12,18,581774,581774,2025-04-12,2026-05-21,0.0
ISOM,29,45,581774,581774,2025-04-12,2026-05-21,0.0
PDA,36,28,581774,581774,2025-04-12,2026-05-21,0.0
LOUP,51,79,537133,581774,2025-04-12,2026-05-21,0.0
PG,16,38,537133,581774,2025-04-12,2026-05-21,0.0
ISOM_3,14,14,568813,581774,2025-04-12,2026-05-21,0.0


Units: 13 | drums: 301 | unique tags: 376 | tag-drum slots: 474
Overall time range: 2025-04-12 00:00:00 -> 2026-05-21 00:00:00 | rows per tag: 321128-581774
Units dropped (no parquet data): ['CPP', 'LPG', 'PDU']
